In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t047'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is related to the "cart" service operation `POST /oteldemo.CartService/AddItem`. The issue is a response time fluctuation, and the alert occurred between 2026-04-04 22:37:09 and 2026-04-04 22:43:03 (UTC+8). The user is requesting help to analyze the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

('', {'start': '2026-04-04T22:37:09.273091+08:00', 'end': '2026-04-04T22:43:03+08:00'}, {'entity_id': 'e433b3f842dc4e525a834701cc603c28', 'entity_name': 'cart::POST /oteldemo.CartService/AddItem', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">cart响应时间抖动告警<alert_event event_id="ca9e5116885d6621592df45c9595c93f" trans_id="eu5vqogil96kucd7f2fqdvk2u6" rule_id="40ef4477-f6e2-414e-8307-1009a8e45c35" rule_name="cart响应时间抖动告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_df = pd.read_parquet(metrics_file_path)

# List all unique KPIs and entity sets
unique_kpis = metrics_df['metric'].unique()
unique_entity_sets = metrics_df['entity_set'].unique()

unique_kpis, unique_entity_sets

```
Out[2]:
```
The metrics data contains various KPIs such as `error_rate`, `latency`, `request_count`, `slow_count`, `node_cpu_usage_rate`, `deployment_memory_usage_total`, and many others. The entity sets include `k8s.node`, `k8s.deployment`, `k8s.cluster`, `k8s.namespace`, `apm.operation`, `apm.instance`, `apm.metric.jvm`, `apm.metric.thread`, and `apm.service.legacy`. These KPIs and entity sets will help in analyzing the performance and issues across different components of the system.

The original code execution output of IPython Kernel is also provided below for reference:

(array(['node_ready_status', 'node_memory_usage_rate',
       'node_not_ready_status', 'node_pod_running_count',
       'node_disk_usage_rate', 'node_cpu_usage_rate',
       'deployment_ready_replicas', 'deployment_available_replicas',
       'deployment_availability_rate', 'deployment_desired_replicas',
       'deployment_memory_usage_total', 'deployment_cpu_usage_total',
       'deployment_cpu_usage_vs_limits',
       'deployment_cpu_usage_vs_requests', 'cluster_node_count',
       'cluster_failed_node_count', 'apiserver_non_2xx_write_request_qps',
       'apiserver_concurrent_connections', 'cluster_number_of_pod',
       'apiserver_non_2xx_read_request_qps',
       'cluster_number_of_not_ready_pods',
       'apiserver_requests_per_second',
       'namespace_number_of_not_ready_pods', 'namespace_number_of_pods',
       'error_count', 'slow_count', 'request_count',
       'avg_request_latency_seconds', 'arms_jvm_gc_total',
       'arms_jvm_gc_seconds_total', 'arms_jvm_gc_delta',
       'arms_jvm_gc_seconds_delta', 'arms_jvm_mem_used_bytes',
       'arms_jvm_mem_init_bytes', 'arms_jvm_mem_committed_bytes',
       'arms_jvm_mem_max_bytes', 'arms_jvm_threads_count',
       'arms_thread_pool_active_thread_count',
       'arms_thread_pool_current_thread_count',
       'arms_thread_pool_max_thread_count',
       'arms_thread_pool_max_pool_size',
       'arms_thread_pool_core_pool_size',
       'arms_thread_pool_scheduled_task_count',
       'arms_thread_pool_completed_task_count',
       'arms_thread_pool_rejected_task_count',
       'arms_thread_pool_queue_size', 'arms_grouped_thread_count',
       'arms_grouped_thread_cpu_time',
       'arms_grouped_thread_runnable_count',
       'arms_grouped_thread_waiting_count',
       'arms_grouped_thread_timed_waiting_count', 'cpu', 'latency', 'mem',
       'workload', 'error', 'slow', 'error_rate'], dtype=object), array(['k8s.node', 'k8s.deployment', 'k8s.cluster', 'k8s.namespace',
       'apm.operation', 'apm.instance', 'apm.metric.jvm',
       'apm.metric.thread', 'apm.service.legacy'], dtype=object))```
```

In [ ]:
# In[3]:

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Create a new column to classify by 'component-KPI'
metrics_df['component_kpi'] = metrics_df['entity_name'] + '-' + metrics_df['metric']

# Calculate the global P95 threshold for each 'component-KPI' time series
p95_thresholds = metrics_df.groupby('component_kpi')['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['component_kpi', 'p95_threshold']

p95_thresholds

```
Out[3]:
```
The global P95 thresholds have been calculated for each 'component-KPI' time series across the dataset. These thresholds represent the 95th percentile of metric values for each combination of component and KPI, helping to identify outliers or abnormal behavior. There are 298 unique 'component-KPI' combinations in total. If you need to examine specific components or KPIs in detail, let me know.

The original code execution output of IPython Kernel is also provided below for reference:

component_kpi  p95_threshold
0                    -apiserver_concurrent_connections       5.000000
1                  -apiserver_non_2xx_read_request_qps       0.088368
2                 -apiserver_non_2xx_write_request_qps       0.088254
3                       -apiserver_requests_per_second      26.431767
4                           -cluster_failed_node_count       0.000000
..                                                 ...            ...
293                                  shipping-workload   12387.750000
294  shipping::oteldemo.ShippingService/GetQuote-av...       0.002892
295  shipping::oteldemo.ShippingService/GetQuote-re...    6197.750000
296  shipping::oteldemo.ShippingService/ShipOrder-a...       0.000009
297  shipping::oteldemo.ShippingService/ShipOrder-r...    6189.250000

[298 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pytz
from datetime import datetime

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-04T22:37:09.273091+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-04T22:43:03+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the alert time window to microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter the metrics data to include only the time window of the alert
filtered_metrics = metrics_df[(metrics_df['time'] >= alert_start_us) & (metrics_df['time'] <= alert_end_us)]

# Merge the filtered metrics with the global P95 thresholds
anomalies = filtered_metrics.merge(p95_thresholds, on='component_kpi', how='left')

# Identify anomalies where the metric value exceeds the P95 threshold
anomalies = anomalies[anomalies['value'] > anomalies['p95_threshold']]

anomalies

```
Out[4]:
```
During the alert time window (2026-04-04 22:37:09 to 2026-04-04 22:43:03 UTC+8), 690 anomalies were identified where metric values exceeded their respective global P95 thresholds. These anomalies span various components and KPIs, including high `node_memory_usage_rate` and elevated `error_rate` for services like `frontend`, `frontend-proxy`, `cart`, and `inventory`. These anomalies indicate potential performance issues or errors in these components during the alert period. Further analysis is needed to pinpoint the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

time domain     entity_set                         entity_id                      entity_name  ...      value                                     metric_set_id service                               component_kpi p95_threshold
91    1775313549000000    k8s       k8s.node                                                                     ...  91.363982  k8s@metric_set@k8s.metric.high_level_metric_node    None                     -node_memory_usage_rate     91.344888
92    1775313579000000    k8s       k8s.node                                                                     ...  91.378649  k8s@metric_set@k8s.metric.high_level_metric_node    None                     -node_memory_usage_rate     91.344888
93    1775313609000000    k8s       k8s.node                                                                     ...  91.481706  k8s@metric_set@k8s.metric.high_level_metric_node    None                     -node_memory_usage_rate     91.344888
94    1775313639000000    k8s       k8s.node                                                                     ...  91.571014  k8s@metric_set@k8s.metric.high_level_metric_node    None                     -node_memory_usage_rate     91.344888
95    1775313669000000    k8s       k8s.node                                                                     ...  91.521729  k8s@metric_set@k8s.metric.high_level_metric_node    None                     -node_memory_usage_rate     91.344888
...                ...    ...            ...                               ...                              ...  ...        ...                                               ...     ...                                         ...           ...
7162  1775313759000000    apm  apm.operation  9e69280c47b4c2f35576402946e3b714         frontend::POST /api/cart  ...   0.977475           apm@metric_set@apm.metric.apm.operation    None         frontend::POST /api/cart-error_rate      0.966847
7185  1775313639000000    apm   apm.instance  d1f1b0349fadcfdc23ca3918ac448623  frontend-proxy-67d64dc4f8-xz2fd  ...   0.884289            apm@metric_set@apm.metric.apm.instance    None  frontend-proxy-67d64dc4f8-xz2fd-error_rate      0.881240
7198  1775313759000000    apm   apm.instance  edd1b3a198ca77c10c1e4414044e4671            cart-67f957cc88-njsn7  ...   0.907801            apm@metric_set@apm.metric.apm.instance    None            cart-67f957cc88-njsn7-error_rate      0.883064
7207  1775313759000000    apm   apm.instance  8709d171a640d6264396ee9b86b3604c            cart-67f957cc88-q9w2g  ...   0.871846            apm@metric_set@apm.metric.apm.instance    None            cart-67f957cc88-q9w2g-error_rate      0.868135
7218  1775313759000000    apm   apm.instance  2da48118270a22f4de7ac642102def69        inventory-887d8db89-7v4s5  ...   0.245810            apm@metric_set@apm.metric.apm.instance    None        inventory-887d8db89-7v4s5-error_rate      0.239787

[690 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Sort anomalies by component_kpi and time for consecutive sub-series analysis
anomalies = anomalies.sort_values(by=['component_kpi', 'time'])

# Identify consecutive sub-series where metric values consistently exceed the P95 threshold
anomalies['is_consecutive'] = (anomalies['component_kpi'] != anomalies['component_kpi'].shift()) | \
                              (anomalies['time'] - anomalies['time'].shift() > 1_000_000 * 60)  # 1-minute gap
anomalies['fault_group'] = anomalies['is_consecutive'].cumsum()

# Group by component_kpi and fault_group to filter out isolated noise spikes
faults = anomalies.groupby(['component_kpi', 'fault_group']).filter(lambda x: len(x) > 1)

# Determine which components and KPIs experienced faults
faulty_components_kpis = faults[['component_kpi']].drop_duplicates()

faulty_components_kpis

```
Out[5]:
```
During the alert time window, 120 distinct 'component-KPI' combinations experienced faults where metric values consistently exceeded the global P95 thresholds. These include issues like high `apiserver_concurrent_connections`, elevated `apiserver_requests_per_second`, and anomalies in components such as `recommendation` and `shipping`. These faults indicate sustained performance issues or errors in these components and KPIs, requiring further investigation to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

component_kpi
4319                  -apiserver_concurrent_connections
4308               -apiserver_non_2xx_write_request_qps
4367                     -apiserver_requests_per_second
4354                  -cluster_number_of_not_ready_pods
3449                        -deployment_cpu_usage_total
...                                                 ...
5599       recommendation-95bcd75df-rxwhh-request_count
6903                                 recommendation-cpu
6541                            recommendation-workload
4889  recommendation::/oteldemo.RecommendationServic...
5045  shipping::oteldemo.ShippingService/ShipOrder-a...

[120 rows x 1 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_df = pd.read_parquet(traces_file_path)

# Ensure 'startTime', 'endTime', and 'duration' columns are numeric for filtering and analysis
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')

# Convert the alert time window to nanoseconds for filtering
alert_start_ns = int(alert_start.timestamp() * 1_000_000_000)
alert_end_ns = int(alert_end.timestamp() * 1_000_000_000)

# Filter traces for the 'cart' service operation during the alert time window
cart_traces = traces_df[
    (traces_df['serviceName'] == 'cart') &
    (traces_df['spanName'] == 'POST /oteldemo.CartService/AddItem') &
    (traces_df['startTime'] >= alert_start_ns) &
    (traces_df['endTime'] <= alert_end_ns)
]

# Analyze the call chain to identify the most downstream faulty service
# Sort by duration to prioritize slow traces
cart_traces_sorted = cart_traces.sort_values(by='duration', ascending=False)

# Extract the most downstream faulty service by analyzing the call chain
downstream_faulty_service = cart_traces_sorted[['traceId', 'spanName', 'serviceName', 'duration', 'statusCode', 'statusMessage']]

downstream_faulty_service

```
Out[6]:
```
The traces for the `POST /oteldemo.CartService/AddItem` operation in the `cart` service during the alert time window reveal consistent failures. The most downstream faulty service is the `inventory` service, as indicated by the error message "Connection refused (inventory:9090)" in all traces. This suggests that the `cart` service's issues are caused by its inability to connect to the `inventory` service.

The original code execution output of IPython Kernel is also provided below for reference:

traceId                            spanName serviceName     duration statusCode                        statusMessage
327214  82e8e948752c174f74327c0a11f755a4  POST /oteldemo.CartService/AddItem        cart  78776363300          2  Connection refused (inventory:9090)
327238  65c45ed51d8e378e53c989fb6ce3f7ec  POST /oteldemo.CartService/AddItem        cart  76937651600          2  Connection refused (inventory:9090)
327236  ab46d49b2b3cde8d91286166db24ed13  POST /oteldemo.CartService/AddItem        cart  76884829000          2  Connection refused (inventory:9090)
327206  e5bf749e65693dbcb3594ff7839fa5c8  POST /oteldemo.CartService/AddItem        cart  76325781000          2  Connection refused (inventory:9090)
322981  3e769ee2f601c06a4342254696071f30  POST /oteldemo.CartService/AddItem        cart  76298220800          2  Connection refused (inventory:9090)
...                                  ...                                 ...         ...          ...        ...                                  ...
390028  3782807f7e0137bdc8764bcc3a1f3373  POST /oteldemo.CartService/AddItem        cart      4810600          2  Connection refused (inventory:9090)
404930  a2f8f3faeae157e2ca9c5c1de6ef0f10  POST /oteldemo.CartService/AddItem        cart      4590600          2  Connection refused (inventory:9090)
404924  2b8795cb044394fc38a3c0bde01e36b6  POST /oteldemo.CartService/AddItem        cart      4313000          2  Connection refused (inventory:9090)
389943  3f8d47c04560f7d172b2c8470eda4aef  POST /oteldemo.CartService/AddItem        cart      4128300          2  Connection refused (inventory:9090)
409155  dac31dfcdfc4d5da44127fd07a79a937  POST /oteldemo.CartService/AddItem        cart      3951500          2  Connection refused (inventory:9090)

[798 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_df = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is in datetime format for filtering
logs_df['_time_'] = pd.to_datetime(logs_df['_time_'], errors='coerce')

# Filter logs for the 'inventory' service during the alert time window
inventory_logs = logs_df[
    (logs_df['_time_'] >= alert_start) &
    (logs_df['_time_'] <= alert_end) &
    (logs_df['_container_name_'].str.contains('inventory', case=False, na=False))
]

# Analyze logs for error messages or indications of issues
inventory_error_logs = inventory_logs[inventory_logs['content'].str.contains('error|fail|exception', case=False, na=False)]

inventory_error_logs

```
Out[7]:
```
The logs for the `inventory` service during the alert time window contain error-related entries, including debug messages and Java-related options. However, the specific error messages do not provide clear indications of the root cause for the connection failures. Further investigation into the service's configuration, resource usage, or upstream dependencies may be required to identify the issue.

The original code execution output of IPython Kernel is also provided below for reference:

content                              _time_ _source_ _container_ip_                                       _image_name_  ... __tag__:__pack_id__    __tag__:__hostname__     __tag__:_node_name_ __tag__:_node_ip_               __tag__:_cluster_id_
513125  2026-04-04 14:42:09 [main] DEBUG [,] o.s.w.s.m... 2026-04-04 22:42:09.379937154+08:00   stdout    10.0.16.157  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  408366E4B9E3FFA3-9  cn-hongkong.10.0.16.64  cn-hongkong.10.0.16.64        10.0.16.64  cfbbc0eabc19d43c0a6fb6889b4451ad0
582990  Picked up JAVA_TOOL_OPTIONS:  -javaagent:/home... 2026-04-04 22:42:11.045433251+08:00   stderr      10.0.2.36  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  9BD1BDF962F3E191-1   cn-hongkong.10.0.2.37   cn-hongkong.10.0.2.37         10.0.2.37  cfbbc0eabc19d43c0a6fb6889b4451ad0
586489  2026-04-04 14:42:16 [main] DEBUG [,] o.s.w.s.m... 2026-04-04 22:42:16.697388510+08:00   stdout    10.0.16.184  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  EF1F6C14A511EE8E-8   cn-hongkong.10.0.2.71   cn-hongkong.10.0.2.71         10.0.2.71  cfbbc0eabc19d43c0a6fb6889b4451ad0
595135  2026-04-04 14:42:39 [main] DEBUG [,] o.s.w.s.m... 2026-04-04 22:42:39.193817306+08:00   stdout      10.0.2.36  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  9BD1BDF962F3E191-9   cn-hongkong.10.0.2.37   cn-hongkong.10.0.2.37         10.0.2.37  cfbbc0eabc19d43c0a6fb6889b4451ad0

[4 rows x 15 columns]```
```

In [ ]:
# In[8]:

# Filter the metrics data for the 'inventory' service during the alert time window
inventory_metrics = metrics_df[
    (metrics_df['entity_name'].str.contains('inventory', case=False, na=False)) &
    (metrics_df['time'] >= alert_start_us) &
    (metrics_df['time'] <= alert_end_us)
]

# Analyze KPIs for anomalies or faults
inventory_kpi_analysis = inventory_metrics.groupby(['metric']).agg(
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    mean_value=('value', 'mean'),
    count=('value', 'count')
).reset_index()

inventory_kpi_analysis

```
Out[8]:
```
During the alert time window, the `inventory` service exhibited several anomalies in its KPIs. Notable observations include:

- High `error_rate` (up to 54.4%) and `error_count` (up to 44), indicating frequent errors.
- Elevated `latency` (up to 14.25 seconds) and `avg_request_latency_seconds` (up to 16.73 seconds), suggesting significant delays in processing requests.
- High `slow_count` (up to 349) and `slow` (up to 724), indicating many slow operations.
- Resource usage metrics like `cpu` (up to 97.4%) and `mem` (up to 1.11 GB) also show high utilization.

These anomalies suggest that the `inventory` service was under significant stress or experiencing faults, which likely caused the connection failures observed in the `cart` service.

The original code execution output of IPython Kernel is also provided below for reference:

metric     min_value     max_value    mean_value  count
0                 arms_grouped_thread_count  1.799051e+00  4.365031e+00  3.235149e+00      5
1              arms_grouped_thread_cpu_time  1.764000e+10  7.130000e+10  4.183800e+10      5
2        arms_grouped_thread_runnable_count  4.620253e-01  2.804703e+00  1.590720e+00      5
3   arms_grouped_thread_timed_waiting_count  7.924335e-01  1.135881e+00  9.201386e-01      5
4         arms_grouped_thread_waiting_count  4.217791e-01  6.284501e-01  5.412753e-01      5
5                         arms_jvm_gc_delta  1.540000e+02  5.350000e+02  3.384000e+02      5
6                 arms_jvm_gc_seconds_delta  2.041000e+01  8.848400e+01  4.550940e+01      5
7                 arms_jvm_gc_seconds_total  2.591200e+01  4.951990e+02  2.080796e+02      5
8                         arms_jvm_gc_total  5.950000e+02  1.189510e+05  3.259880e+04      5
9              arms_jvm_mem_committed_bytes  3.858691e+08  7.156750e+08  5.262777e+08      5
10                  arms_jvm_mem_init_bytes  4.122214e+07  4.122214e+07  4.122214e+07      5
11                   arms_jvm_mem_max_bytes  1.862271e+09  1.862271e+09  1.862271e+09      5
12                  arms_jvm_mem_used_bytes  3.402304e+08  6.459592e+08  4.724319e+08      5
13                   arms_jvm_threads_count  3.091667e+01  6.206204e+02  2.105691e+02      5
14     arms_thread_pool_active_thread_count  1.785714e+00  1.614352e+01  9.613797e+00      5
15    arms_thread_pool_completed_task_count  2.940000e+02  4.811000e+03  1.619400e+03      5
16          arms_thread_pool_core_pool_size  1.857143e+00  2.000000e+00  1.903333e+00      5
17    arms_thread_pool_current_thread_count  3.702381e+00  1.773148e+01  1.116101e+01      5
18           arms_thread_pool_max_pool_size  1.073742e+09  1.227134e+09  1.147804e+09      5
19        arms_thread_pool_max_thread_count  4.077381e+00  1.774074e+01  1.128553e+01      5
20              arms_thread_pool_queue_size  1.726190e+00  1.155238e+01  5.648644e+00      5
21     arms_thread_pool_rejected_task_count -6.000000e+00  4.900000e+01  1.800000e+01      5
22    arms_thread_pool_scheduled_task_count  5.810000e+02  5.124000e+03  2.168800e+03      5
23              avg_request_latency_seconds  0.000000e+00  1.672928e+01  4.746117e+00     66
24                                      cpu  2.952269e-02  9.740602e-01  4.791036e-01     11
25                                    error  1.000000e+00  4.400000e+01  1.272727e+01     11
26                              error_count  1.000000e+00  4.400000e+01  1.800000e+01     10
27                               error_rate  3.424658e-03  5.443038e-01  1.943431e-01     10
28                                  latency  1.256883e-03  1.425327e+01  8.190261e+00     11
29                                      mem  4.312405e+08  1.112381e+09  7.623217e+08     11
30                            request_count  1.000000e+00  4.558000e+03  4.142727e+02     66
31                                     slow  1.150000e+02  7.240000e+02  3.912727e+02     11
32                               slow_count  1.000000e+00  3.490000e+02  9.396226e+01     53
33                                 workload  1.800000e+02  4.613000e+03  1.401091e+03     11```
```